# M1 · 02 — Official validation and parity diagnosis (CPU)

**Purpose.** Prove that the champion reproduces its validation behaviour with the existing official scorer, and that the map that will be exported is the map that was evaluated. Diagnose the historical W2 parity blocker from evidence.

| | |
|---|---|
| Runtime | CPU, High-RAM (the dry zone-stitched build holds full-grid rasters) |
| Prerequisites | notebook 01 status COMPLETE |
| Inputs | `selection/*`, the frozen grids (`boost/benchmark/grid.csv`, `stack/grid.csv`), W2 / CNN held-out maps, `recovery/export_parity/train_inference_parity.csv`, `stack/maps/full_score.tif` (historical, optional) |
| Outputs | `validation/{official_scores, per_zone_metrics, weighted_components, recorded_reproduction, zone_budget}.csv`, `validation/validation_status.json`, `parity/{parity_report.csv, parity_report.json, historical_parity_summary.csv, zone_stitched_parity.csv}`, `figures/*.png` + `_data.csv`, `reports/validation_report.md` |
| Next | `03_SUBMISSION_EXPORT.ipynb` when validation AND parity are COMPLETE |

**Tolerances** (`configs/leaderboard_focus/m1.yaml` → `validation`):
* recomputed components vs saved grid: rtol / atol 1e-6 (same code path, float rounding only);
* pooled DTI: 1e-6;
* recorded values: 6e-6 (they were stored with 6 decimals);
* zone-score parity: Spearman ≥ 0.999. In strict mode any non-PASS row blocks, including the fusion-weight WARNING.

A value outside its tolerance makes the stage **BLOCKED**.

## 1 · Configuration

In [ ]:
CONFIG = {'repo': 'mybenkhadda/GEMS---drivendata', 'ref': 'main', 'drive_root': '/content/drive/MyDrive/GEMS_2',
          'local_root': '/content/gems_work', 'reuse_cached': True, 'strict_provenance': True,
          'overwrite_existing': False, 'device': 'cpu'}
PRIVATE_REPO = False
M1_CONFIG = {}
EXPECT = {'code_rev': 'm1.1-r1', 'commit': None}

## 2 · Drive, strict clone, version lock, shared session

In [ ]:
import base64, os, subprocess, sys, json
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
def git(*a, check=True):
    env = None
    if PRIVATE_REPO:
        from google.colab import userdata
        hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
        env = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}
    r = subprocess.run(['git', *a], capture_output=True, text=True, env=env)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(a[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r
if not Path(CODE, '.git').exists():
    git('clone', '-q', 'https://github.com/' + CONFIG['repo'] + '.git', CODE)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin')
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + CONFIG['ref'], check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'm1_colab_active', 'origin/' + CONFIG['ref'])
else:
    git('-C', CODE, 'checkout', '-q', '--detach', CONFIG['ref'])
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules:
    raise RuntimeError('gems_bench already imported: restart the runtime')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD
import gems_bench.leaderboard_focus as LF
assert LF.M1_CODE_REV == EXPECT['code_rev'] and (not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']))
from gems_bench.leaderboard_focus.session import M1Session
m1 = M1Session.attach(CONFIG, M1_CONFIG)
print('HEAD', HEAD, '| selection status:', m1.status('selection'))

## 3 · Official validation

Steps:
1. Recompute each held-out cell from the frozen maps and compare it with the saved grid row.
2. Check that the recomputed values pool to the nested summary.
3. Reproduce the recorded values from the frozen grids.
4. Check the template against the feature grid.
5. Run the zone-stitched dry build (no file written) and check parity, coverage, per-zone budget, the known-fault mask and recipe consistency.
6. Record the sha256 of the validated array.

In [ ]:
from IPython.display import Image, Markdown, display
from gems_bench.leaderboard_focus.validation import validate
V = validate(m1)
VS = m1.ws.read_json('validation/validation_status.json') or {}
print('VALIDATION STATUS:', V['status'])
for r in V['reasons']:
    print('  reason:', r)
display(m1.ws.load_table('validation/official_scores.csv'))
display(m1.ws.load_table('validation/recorded_reproduction.csv'))
print('validated array sha256:', VS.get('validated_array_sha256'))

## 4 · Parity diagnosis

The report keeps three evidence sources apart:
* **historical**: the recorded P6.1 table, classified by cause (rank domain, score map, by design, representation / budget, fusion weight);
* **recomputed**: cheap checks on frozen files (score-map agreement with the historical export, fold assignment, duplicated / missing pixels, the historical export recipe, OOF vs in-sample);
* **champion**: the dry-build parity.

Only champion rows can block M1.

In [ ]:
from gems_bench.leaderboard_focus.parity import diagnose
PRT = diagnose(m1)
PJ = m1.ws.read_json('parity/parity_report.json') or {}
print('PARITY STATUS:', PRT['status'])
for r in PRT['reasons']:
    print('  reason:', r)
print(json.dumps(PJ.get('historical_blocker'), indent=2, default=str)[:3000])
display(m1.ws.load_table('parity/parity_report.csv')[['scope', 'check', 'unit', 'status', 'source']])
for n in ('dti_by_region', 'recorded_vs_reproduced', 'area_budget_consistency', 'score_map_parity', 'prediction_coverage'):
    p = m1.ws.path(f'figures/{n}.png')
    if p.exists():
        display(Image(filename=str(p)))

## 5 · Verification checklist

1. `VALIDATION STATUS: COMPLETE` and `PARITY STATUS: COMPLETE`.
2. `per_zone_metrics.csv`: every row PASS. `recorded_reproduction.csv`: every row PASS.
3. `zone_budget.csv`: every zone `ok` (area within 2 % of the policy fraction, 0 positive pixels on mapped faults).
4. `parity_report.json` → `historical_blocker` names the recorded causes; `champion_parity` is all PASS.

## 6 · Troubleshooting

* **Recomputed ≠ saved grid.** A frozen input changed after boost A. Check `selection/champion_provenance.json` hashes against `artifact_inventory.json`. Do not relax the tolerance.
* **Recorded value not reproduced.** The grid on Drive is not the one that produced the record. The stage stays BLOCKED until the right artifact is restored.
* **Zone parity WARNING (fusion w).** The champion blends the fusion with a per-zone weight in evaluation but an all-zone weight at deployment. Strict mode blocks it. Either a blend-1.0 champion passes, or you decide explicitly with `M1_CONFIG = {'validation': {'strict_parity': False}}`, which is recorded.
* **Out of memory.** Use a High-RAM runtime.

## 7 · Next

`03_SUBMISSION_EXPORT.ipynb`.

In [ ]:
display(Markdown(m1.ws.path('reports/validation_report.md').read_text(encoding='utf-8')))